# 🤖 GitHub Copilot SDK 미니 워크샵 


**예상 전체 실습 시간: 약 1시간**

## 🎯 학습 목표

이 워크샵을 마치면 다음을 이해할 수 있습니다.

- CopilotClient와 Session의 역할
- send_and_wait와 이벤트 스트리밍의 차이
- @define_tool로 에이전트 도구를 만드는 방법
- 시스템 프롬프트로 에이전트 행동을 설계하는 방법
- FastAPI와 SSE로 브라우저에 실시간 스트리밍하는 방법
- 분석 결과를 GitHub 이슈에 다시 기록하는 방법
- 안전 훅으로 도구 사용을 검증하는 기본 아이디어

권장 실행 방식은 노트북에서 개념과 코드를 따라가고, 실제 서버 실행은 터미널에서 진행하는 방식입니다. 💡

## 🗺️ 전체 흐름

이번 실습의 큰 흐름은 아래와 같습니다.

1. Copilot SDK 클라이언트를 시작합니다.
2. 세션을 생성하고 모델, 도구, 권한 정책을 연결합니다.
3. 에이전트가 GitHub API 도구를 호출하며 이슈와 코드를 조사합니다.
4. 결과를 CLI 또는 웹 UI로 스트리밍합니다.
5. 필요하면 사람이 검토한 뒤 GitHub에 다시 코멘트와 라벨을 남깁니다.

이제 단계별로 구현해보겠습니다. ✨

## 1단계. 기본 import와 환경 준비 📦

먼저 전체 실습에서 공통으로 사용할 모듈을 가져옵니다.

### 코드 설명

- asyncio: 비동기 함수 실행
- base64, json: GitHub API 응답 파싱
- Path: 정적 파일 경로 처리
- load_dotenv: .env 파일에서 토큰 읽기
- BaseModel, Field: 도구 파라미터 스키마 정의
- CopilotClient, define_tool: Copilot SDK 핵심 구성요소
- PermissionHandler: 데모에서 도구 권한 자동 승인

In [6]:
import asyncio
import base64
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from copilot import CopilotClient, define_tool
from copilot.session import PermissionHandler

load_dotenv()

token = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
print("GitHub PAT 상태:", "성공 ✅" if token else "미설정 ⚠️")

GitHub PAT 상태: 성공 ✅


## 2단계. 간단한 Hello World 만들기 👋

이 단계에서는 send_and_wait를 사용해 가장 단순한 SDK 호출을 해봅니다.

### 핵심 개념

- Copilot Client를 만든다
- Session을 만든다
- 프롬프트를 보내고 전체 응답을 한 번에 받는다

send_and_wait는 구현이 단순해서 입문용으로 좋지만, 토큰이 생성되는 과정을 실시간으로 보여주지는 않습니다.

In [7]:
# Copilot SDK를 사용할 클라이언트 함수를 정의합니다.
async def hello_world():
    """가장 단순한 Copilot SDK 호출 예제"""
    # Copilot 백엔드와 통신할 클라이언트 객체를 생성합니다.
    client = CopilotClient()
    # 클라이언트를 시작해 세션 생성이 가능한 상태로 만듭니다.
    await client.start()

    # 환경 변수에서 GitHub 토큰을 읽어 옵니다.
    token = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
    # 사용할 모델, 권한 처리 방식, 토큰 정보를 포함한 세션을 생성합니다.
    session = await client.create_session(
        # 이번 예제에서 사용할 모델 이름입니다.
        model="gpt-4.1",
        # 데모 목적상 툴 권한 요청은 자동 승인합니다.
        on_permission_request=PermissionHandler.approve_all,
        # GitHub 인증 토큰을 세션에 전달합니다.
        github_token=token,
    )

    # 모델에게 질문을 보내고 전체 응답이 완성될 때까지 기다립니다.
    response = await session.send_and_wait(
        "GitHub Copilot SDK가 무엇인지 간략하게 설명해 줘"
    )

    # 정상 응답이 왔고 본문 콘텐츠가 있으면 화면에 출력합니다.
    if response and getattr(response, "data", None) and hasattr(response.data, "content"):
        # 모델이 생성한 최종 텍스트를 출력합니다.
        print(response.data.content)

    # 사용이 끝난 세션 연결을 정리합니다.
    await session.disconnect()
    # 클라이언트도 종료해 리소스를 해제합니다.
    await client.stop()

# 노트북 환경에서는 아래 한 줄로 방금 만든 비동기 함수를 실행할 수 있습니다.
await hello_world()

GitHub Copilot SDK는 개발자가 자신의 애플리케이션이나 서비스에 Copilot의 AI 기능(코드 자동완성, 자연어 명령 처리 등)을 직접 통합할 수 있도록 지원하는 소프트웨어 개발 키트입니다. 이를 통해 Copilot의 AI를 웹, 데스크톱, 커스텀 툴 등 다양한 환경에서 활용할 수 있습니다. 쉽게 말해, Copilot의 AI를 내 앱에 직접 넣을 수 있게 해주는 도구입니다.


### 코드 해설 📝

- client.start()는 Copilot 백엔드와 통신을 시작합니다.
- create_session()에서 모델, 토큰, 권한 정책을 지정합니다.
- send_and_wait()는 응답이 모두 완성될 때까지 기다렸다가 결과를 돌려줍니다.
- 마지막의 disconnect()와 stop()은 리소스 정리에 중요합니다.

터미널에서는 다음처럼 실행합니다.

**python app_final.py hello**

## 3단계. 이벤트 기반 스트리밍 보기 🌊

이번에는 응답이 완성될 때까지 기다리지 않고, 응답이 생성되는 순간마다 받아봅니다.

### 왜 중요한가요?

실전 UI에서는 사용자가 실시간으로 진행 상황을 보는 것이 중요합니다.   
특히 도구 호출이 많은 에이전트는 스트리밍이 체감 품질을 크게 높여줍니다.

In [14]:
# 이벤트 기반 스트리밍 예제를 위한 비동기 함수를 정의합니다.
async def hello_world_streaming():
    """이벤트를 이용해 응답을 실시간으로 스트리밍하는 예제"""
    # Copilot 백엔드와 통신할 클라이언트 객체를 생성합니다.
    client = CopilotClient()
    # 클라이언트를 시작해 세션 생성과 메시지 전송이 가능한 상태로 만듭니다.
    await client.start()

    # 환경 변수에서 GitHub 인증 토큰을 읽어 옵니다.
    token = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
    # 스트리밍을 받을 세션을 생성합니다.
    session = await client.create_session(
        # 이번 예제에서 사용할 모델입니다.
        model="gpt-4.1",
        # 데모에서는 도구 권한 요청을 자동 승인합니다.
        on_permission_request=PermissionHandler.approve_all,
        # GitHub 토큰을 세션에 연결합니다.
        github_token=token,
    )

    # 응답 완료 시점을 기다리기 위한 비동기 이벤트 객체를 만듭니다.
    done = asyncio.Event()

    # 세션에서 발생하는 이벤트를 처리할 콜백 함수를 정의합니다.
    def on_event(event):
        # 이벤트 타입 값을 문자열로 정규화해 비교하기 쉽게 만듭니다.
        event_name = event.type.value if hasattr(event.type, "value") else str(event.type)

        # 모델이 새 텍스트 조각을 보낼 때마다 즉시 출력합니다.
        if event_name == "assistant.message":
            # 줄바꿈 없이 이어 붙여서 실시간 스트리밍처럼 보이게 합니다.
            print(event.data.content, end="", flush=True)
        # 세션이 유휴 상태가 되면 이번 응답이 끝났다는 뜻입니다.
        elif event_name == "session.idle":
            # 대기 중인 done 이벤트를 완료 상태로 바꿉니다.
            done.set()

    # 메시지를 보내기 전에 이벤트 핸들러를 세션에 등록합니다.
    session.on(on_event)
    # 모델에게 질문을 보내고, 실제 응답 수신은 이벤트 콜백이 담당하게 합니다.
    await session.send("GitHub Copilot SDK가 무엇인지 2문장으로 설명해 줘")
    # session.idle 이벤트가 들어올 때까지 기다립니다.
    await done.wait()

    # 출력 마무리를 위해 마지막에 줄바꿈을 한 번 넣습니다.
    print()
    # 사용이 끝난 세션 연결을 종료합니다.
    await session.disconnect()
    # 클라이언트도 종료해 리소스를 정리합니다.
    await client.stop()

# 노트북에서 직접 실행해 보려면 아래 주석을 해제하면 됩니다.
await hello_world_streaming()

GitHub Copilot SDK는 개발자가 자신의 애플리케이션이나 서비스에 Copilot의 AI 기능을 직접 통합할 수 있도록 지원하는 소프트웨어 개발 키트입니다. 이 SDK를 사용하면 자연어 명령을 코드로 변환하거나, 코드 자동 완성, 코드 리뷰, 문서 생성 등 다양한 Copilot 기능을 앱 내에서 활용할 수 있습니다. REST API, 이벤트 기반 인터페이스, 세션 관리 등 다양한 통합 방식을 제공하며, 보안과 개인정보 보호를 고려한 설계가 특징입니다. 개발자는 Copilot SDK를 통해 맞춤형 AI 개발 도구, 챗봇, 코드 분석기 등 다양한 AI 기반 개발자 경험을 구현할 수 있습니다...........................


### 코드 해설 🧠

- session.on()에 이벤트 핸들러를 등록하면 세션 중 발생하는 이벤트를 받을 수 있습니다.
- assistant.message는 모델의 텍스트 조각이 도착할 때 발생합니다.
- session.idle은 현재 턴이 모두 끝났다는 신호입니다.
- 나중에 도구 호출을 UI에 보여줄 때도 이 이벤트 패턴을 그대로 활용합니다.

터미널 실행 예시:

python app_final.py hello-stream

## 4단계. 에이전트 도구 만들기 🛠️

이제 에이전트가 외부 세계와 상호작용할 수 있도록 GitHub API 기반의 로컬 도구(Tool)를 정의합니다.

이번 단계에서는 다양한 로컬 도구를 그 목적과 용도에 따라 만들고 설명합니다.

1. 공통 GitHub API 헬퍼 함수
2. 이슈 조회용 파라미터 클래스와 도구 함수
3. 저장소 구조 조회용 파라미터 클래스와 도구 함수
4. 코드 검색용 파라미터 클래스와 도구 함수
5. 파일 읽기용 파라미터 클래스와 도구 함수

In [22]:
# 여러 도구가 공통으로 사용할 GitHub REST API 호출 헬퍼를 정의합니다.
def github_api(endpoint: str) -> dict:
    """모든 도구가 공통으로 사용하는 GitHub REST API 헬퍼"""
    # 이 함수 안에서만 사용할 HTTP 클라이언트 라이브러리를 불러옵니다.
    import httpx

    # GitHub API 요청에 사용할 기본 헤더를 준비합니다.
    headers = {
        # GitHub REST API v3 응답 포맷을 요청합니다.
        "Accept": "application/vnd.github.v3+json",
        # 요청을 보낸 클라이언트 이름을 식별용으로 남깁니다.
        "User-Agent": "copilot-workshop-ko",
    }
    # 환경 변수에서 GitHub 인증 토큰을 읽어 옵니다.
    token = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
    # 토큰이 있으면 Authorization 헤더를 추가합니다.
    if token:
        headers["Authorization"] = f"Bearer {token}"

    # 동기 HTTP 클라이언트를 열고 요청을 보냅니다.
    with httpx.Client() as http:
        # endpoint 값을 GitHub API 기본 주소 뒤에 붙여 GET 요청을 보냅니다.
        response = http.get(f"https://api.github.com{endpoint}", headers=headers)
        # 4xx, 5xx 응답이 오면 예외를 발생시킵니다.
        response.raise_for_status()
        # JSON 응답 본문을 파이썬 dict로 변환해 반환합니다.
        return response.json()

### 4-1. GitHub Issue 조회 도구 이해하기 📌

첫 번째 도구는 특정 저장소의 이슈 제목, 본문, 라벨, 작성자, 댓글 일부를 가져옵니다.

이 도구는 에이전트가 문제를 처음 이해할 때 가장 먼저 사용할 가능성이 높습니다.

여기서는 두 부분을 함께 봅니다.

- 입력 파라미터를 정의하는 클래스
- 실제 GitHub API를 호출하는 도구 함수

In [ ]:
# 이슈 조회 도구가 받을 입력값의 구조를 정의합니다.
class GetIssueParams(BaseModel):
    # 저장소 소유자 이름을 받습니다.
    owner: str = Field(description="Repository owner")
    # 저장소 이름을 받습니다.
    repo: str = Field(description="Repository name")
    # 조회할 이슈 번호를 받습니다.
    issue_number: int = Field(description="Issue number")

# 이슈 조회 함수를 Copilot 에이전트가 Tool로서 호출할 수 있도록 상세하게 정의합니다.
@define_tool(description="Fetch a GitHub issue including title, body, labels, and comments")
async def get_github_issue(params: GetIssueParams) -> str:
    # 네트워크 오류나 권한 오류가 날 수 있으므로 예외 처리를 감쌉니다.
    try:
        # 이슈 본문 정보를 GitHub Issues API에서 가져옵니다.
        issue = github_api(f"/repos/{params.owner}/{params.repo}/issues/{params.issue_number}")
        # 같은 이슈의 댓글 목록도 별도 API로 가져옵니다.
        comments = github_api(
            f"/repos/{params.owner}/{params.repo}/issues/{params.issue_number}/comments"
        )
        # 모델이 읽기 쉬운 구조로 필요한 정보만 정리해 문자열로 반환합니다.
        return str({
            # 이슈 제목입니다.
            "title": issue["title"],
            # 본문이 비어 있으면 기본 문구를 넣습니다.
            "body": issue.get("body", "No description"),
            # 라벨 객체 목록에서 이름만 뽑아냅니다.
            "labels": [label["name"] for label in issue.get("labels", [])],
            # 작성자 로그인 이름을 담습니다.
            "user": issue["user"]["login"],
            # 댓글은 너무 길어지지 않도록 앞부분 일부만 5개까지 담습니다.
            "comments": [
                {"user": comment["user"]["login"], "body": comment["body"][:500]}
                for comment in comments[:5]
            ],
        })
    # 문제가 생기면 모델이 이해할 수 있도록 오류 문자열을 돌려줍니다.
    except Exception as exc:
        return f"Error fetching issue: {exc}"

### 4-2. Repository 구조 조회 도구 이해하기 🗂️

두 번째 도구는 저장소의 특정 경로에 어떤 파일과 폴더가 있는지 살펴봅니다.

에이전트는 이 도구를 통해 코드베이스의 큰 구조를 파악하고, 어떤 디렉터리를 더 읽어야 할지 결정합니다.

In [ ]:
# 리포지토리 구조 조회 도구의 입력 스키마를 정의합니다.
class RepoStructureParams(BaseModel):
    # 저장소 소유자 이름입니다.
    owner: str = Field(description="Repository owner")
    # 저장소 이름입니다.
    repo: str = Field(description="Repository name")
    # 조회할 디렉터리 경로이며, 비우면 루트 경로를 뜻합니다.
    path: str = Field(default="", description="Directory path")


# 리포지토리 구조 조회 함수를 Copilot 에이전트가 Tool로서 호출할 수 있도록 상세하게 정의합니다.
@define_tool(description="List the directory contents of a GitHub repository")
async def get_repo_structure(params: RepoStructureParams) -> str:
    # API 요청 실패 가능성을 고려해 예외 처리합니다.
    try:
        # GitHub Contents API를 호출해 지정 경로의 내용을 가져옵니다.
        items = github_api(f"/repos/{params.owner}/{params.repo}/contents/{params.path}")
        # 디렉터리면 여러 항목이 담긴 리스트가 반환됩니다.
        if isinstance(items, list):
            # 폴더와 파일을 구분해서 사람이 읽기 쉬운 문자열 목록으로 바꿉니다.
            return "\n".join(
                f"{'📁' if item['type'] == 'dir' else '📄'} {item['path']}"
                for item in items[:50]
            )
        # 단일 파일이면 파일 경로만 간단히 알려줍니다.
        return f"File: {items['path']}"
    # 실패 시 오류 메시지를 문자열로 반환합니다.
    except Exception as exc:
        return f"Error: {exc}"

### 4-3. 코드 검색 도구 이해하기 🔎

세 번째 도구는 리포지토리 안에서 특정 키워드를 검색합니다.

에이전트는 이 도구를 통해 관련 함수, 클래스, 파일 후보를 빠르게 좁힐 수 있습니다.

In [ ]:
# 코드 검색 도구가 받을 입력값 구조를 정의합니다.
class SearchCodeParams(BaseModel):
    # 저장소 소유자 이름입니다.
    owner: str = Field(description="Repository owner")
    # 저장소 이름입니다.
    repo: str = Field(description="Repository name")
    # 검색에 사용할 키워드입니다.
    query: str = Field(description="Search keywords")


# 리포지토리 코드 검색 함수를 Copilot 에이전트가 Tool로서 호출할 수 있도록 상세하게 정의합니다.
@define_tool(description="Search for code in a GitHub repository")
async def search_code_in_repo(params: SearchCodeParams) -> str:
    # 검색 API 호출 중 발생할 수 있는 예외를 처리합니다.
    try:
        # GitHub Code Search API로 키워드와 리포지토리 범위를 함께 전달합니다.
        results = github_api(
            f"/search/code?q={params.query}+repo:{params.owner}/{params.repo}&per_page=10"
        )
        # 검색 결과에서 파일 경로와 이름만 추려서 간단한 목록으로 만듭니다.
        files = [
            {"path": item["path"], "name": item["name"]}
            for item in results.get("items", [])[:10]
        ]
        # 결과가 있으면 문자열로 반환하고, 없으면 안내 문구를 돌려줍니다.
        return str(files) if files else "No matching code found"
    # 실패 시 오류 문자열을 반환합니다.
    except Exception as exc:
        return f"Error: {exc}"

### 4-4. 파일 읽기 도구 이해하기 📄

네 번째 도구는 특정 파일의 실제 내용을 읽어 옵니다.

이 도구를 사용하면 에이전트는 검색으로 후보를 찾은 뒤, 실제 구현 코드까지 확인할 수 있습니다.

In [ ]:
# 파일 읽기 도구의 입력 스키마를 정의합니다.
class FileContentParams(BaseModel):
    # 저장소 소유자 이름입니다.
    owner: str = Field(description="Repository owner")
    # 저장소 이름입니다.
    repo: str = Field(description="Repository name")
    # 읽을 파일 경로입니다.
    path: str = Field(description="File path within the repository")


# 특정 파일 내용을 읽는 함수를 Copilot 에이전트가 Tool로서 호출할 수 있도록 상세하게 정의합니다.
@define_tool(description="Fetch and read a specific file from a GitHub repository")
async def get_file_content(params: FileContentParams) -> str:
    # 네트워크 오류나 디코딩 오류에 대비해 예외 처리합니다.
    try:
        # GitHub Contents API에서 파일 메타데이터와 내용을 가져옵니다.
        data = github_api(f"/repos/{params.owner}/{params.repo}/contents/{params.path}")
        # GitHub가 base64 인코딩으로 내용을 줄 때만 디코딩을 수행합니다.
        if data.get("encoding") == "base64":
            # base64 문자열을 실제 UTF-8 텍스트로 변환합니다.
            text = base64.b64decode(data["content"]).decode("utf-8")
            # 너무 긴 파일은 앞부분만 남기고 잘라서 반환합니다.
            if len(text) > 5000:
                return text[:5000] + "\n...[truncated]"
            # 길이가 적당하면 전체 텍스트를 반환합니다.
            return text
        # 인코딩 정보가 없으면 원본 content 필드를 그대로 사용합니다.
        return data.get("content", "Unable to decode")
    # 문제가 생기면 오류 문자열을 반환합니다.
    except Exception as exc:
        return f"Error: {exc}"


# 나중 단계에서 세션에 넘길 수 있도록 TOOL 목록을 한 번에 묶어 둡니다.
TOOLS = [get_github_issue, get_repo_structure, search_code_in_repo, get_file_content]

### 4단계 정리 ✅

이제 4단계의 각 도구를 개별 블록으로 나누어 보았습니다.

이렇게 나누면 다음이 더 잘 보입니다.

- 각 클래스는 어떤 입력을 받는가
- 각 함수는 어떤 API를 호출하는가
- 어떤 데이터를 모델에게 다시 돌려주는가
- 왜 예외를 문자열로 반환하는가

이 구조를 이해하고 나면, 다음 5단계에서 `TOOLS` 리스트를 세션에 넘기는 이유도 훨씬 자연스럽게 보입니다.

### 코드 해설 🔍

- BaseModel 클래스들은 도구 입력 스키마입니다. 모델이 어떤 인자를 넘겨야 하는지 명확히 알려줍니다.
- @define_tool은 일반 파이썬 함수를 에이전트가 호출 가능한 도구로 등록합니다.
- 각 도구는 문자열을 반환합니다. 이는 모델이 읽기 쉬운 형태를 유지하기 위한 선택입니다.
- 예외를 그대로 던지지 않고 문자열로 반환하면, 모델이 다른 전략으로 재시도할 수 있습니다.
- 파일 읽기 도구는 5000자까지만 반환해 컨텍스트 과다 사용을 방지합니다.

## 5단계. 시스템 프롬프트와 CLI 분석기 만들기 🧭

이제 TOOL 목록과 시스템 프롬프트를 세션에 연결해, 실제로 이슈를 분석하는 에이전트를 만듭니다.

### 이 단계의 포인트

- TOOOL을 세션에 등록한다
- 시스템 프롬프트로 에이전트의 역할을 지정한다
- 이벤트로 메시지와 도구 호출을 같이 출력한다

In [ ]:
# 세션에 등록할 도구 목록을 한 곳에 모아 둡니다.
TOOLS = [get_github_issue, get_repo_structure, search_code_in_repo, get_file_content]

# 에이전트의 역할, 절차, 응답 형식을 고정하기 위한 시스템 프롬프트입니다.
SYSTEM_PROMPT = """당신은 GitHub 이슈를 분류(triage)하는 시니어 엔지니어링 매니저입니다.

이슈를 분석할 때 다음을 수행합니다:
1. get_github_issue 도구를 사용하여 이슈 세부 정보를 가져옵니다.
2. 코드베이스를 이해하기 위해 리포지토리 구조를 탐색합니다.
3. 관련된 소스 파일을 검색하고 읽습니다.
4. 구조화된 복잡도 평가(assessment)를 제공합니다.

응답 형식은 다음과 같습니다:
## 이슈 요약 (Issue Summary)
## 복잡도 평가 (Complexity Assessment)
- **권장 기술 수준**: Junior / Mid-level / Senior / Senior+
- **신뢰도**: High / Medium / Low
## 분석 근거 (Reasoning)
## 관련 가능 파일 (Files Likely Involved)
## 제안된 접근 방식 (Suggested Approach)
## 멘토링 노트 (Mentorship Notes)

이 이슈를 해결하기 위해 경험이 적은 개발자가 학습해야 할 내용도 포함하세요."""

# 터미널에서 이슈 분석 결과를 스트리밍으로 출력하는 비동기 함수입니다.
async def analyse_cli(owner: str, repo: str, issue_number: int):
    # 함수의 역할을 한 줄 설명으로 남깁니다.
    """터미널에서 분석 결과를 스트리밍 출력하는 함수"""
    # 지금 어떤 저장소의 어떤 이슈를 분석 중인지 사용자에게 먼저 보여줍니다.
    print(f"\n🔍 {owner}/{repo}의 #{issue_number} 이슈 분석 중...\n")

    # Copilot 백엔드와 통신할 클라이언트 객체를 생성합니다.
    client = CopilotClient()
    # 세션 생성 전에 클라이언트를 시작합니다.
    await client.start()

    # GitHub API 호출과 세션 인증에 사용할 토큰을 환경 변수에서 읽습니다.
    token = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
    # 모델, 도구, 권한 정책, GitHub 토큰을 포함한 세션을 생성합니다.
    session = await client.create_session(
        # 이번 실습에서 사용할 모델 이름입니다.
        model="gpt-4.1",
        # 에이전트가 호출할 수 있는 도구 목록을 세션에 등록합니다.
        tools=TOOLS,
        # 데모 목적상 도구 실행 권한 요청은 자동 승인합니다.
        on_permission_request=PermissionHandler.approve_all,
        # GitHub 토큰을 세션에 전달합니다.
        github_token=token,
    )

    # 응답 완료 시점을 기다리기 위한 비동기 이벤트를 만듭니다.
    done = asyncio.Event()

    # 세션에서 발생하는 이벤트를 처리할 콜백 함수를 정의합니다.
    def on_event(event):
        # 이벤트 타입 값을 문자열로 바꿔 비교하기 쉽게 만듭니다.
        event_name = event.type.value if hasattr(event.type, "value") else str(event.type)

        # 모델이 텍스트를 반환할 때마다 즉시 터미널에 이어서 출력합니다.
        if event_name == "assistant.message":
            print(event.data.content, end="", flush=True)
        # 도구 호출이 시작되면 어떤 도구가 실행 중인지 별도로 보여줍니다.
        elif event_name in ("tool.call", "tool.execution_start"):
            # 이벤트 구조 차이를 고려해 도구 이름을 안전하게 꺼냅니다.
            tool_name = getattr(event.data, "name", None) or getattr(event.data, "tool_name", "")
            print(f"\n🔧 {tool_name} 도구 호출 중...", flush=True)
        # 세션이 유휴 상태가 되면 이번 턴이 끝난 것으로 보고 대기를 해제합니다.
        elif event_name == "session.idle":
            done.set()

    # 메시지 전송 전에 이벤트 핸들러를 세션에 등록합니다.
    session.on(on_event)
    # 시스템 프롬프트와 실제 분석 대상 이슈 정보를 함께 전달합니다.
    await session.send(
        f"{SYSTEM_PROMPT}\n\nPlease analyse GitHub issue #{issue_number} in {owner}/{repo}."
    )
    # session.idle 이벤트가 올 때까지 기다립니다.
    await done.wait()

    # 출력 마무리를 위해 줄바꿈을 한 번 추가합니다.
    print()
    # 사용이 끝난 세션 연결을 종료합니다.
    await session.disconnect()
    # 클라이언트도 종료해 리소스를 정리합니다.
    await client.stop()

# 아래 한 줄은 예시 실행이며, 여러분의 실제 저장소와 이슈 번호로 바꿔 실행할 수 있습니다.
# 단순 테스트를 위해서는 아래의 인자 그대로 실행해도 무방합니다. 
await analyse_cli("taeyo-kim", "MyDemo", 78)


🔍 taeyo-kim/MyDemo의 #78 이슈 분석 중...


🔧 get_github_issue 도구 호출 중...

🔧 get_repo_structure 도구 호출 중...
## 이슈 요약 (Issue Summary)
- 블로그 글(Post)에 공개/비공개(visibility) 설정 기능을 추가하여, 작성자가 글의 공개 범주를 선택할 수 있도록 하는 기능 요청입니다. 모델, 폼, 뷰, 템플릿, 테스트까지 전반적인 변경이 요구됩니다.

## 복잡도 평가 (Complexity Assessment)
- **권장 기술 수준**: Mid-level
- **신뢰도**: Medium

## 분석 근거 (Reasoning)
- 모델, 폼, 뷰, 템플릿, 테스트 등 Django의 주요 컴포넌트 전반을 다룹니다.
- 요구사항이 명확하고, 예시 코드와 상세 작업 목록이 제공되어 있습니다.
- 다만, 현재 리포지토리 구조상 Django 관련 파일이 보이지 않아, 실제로는 프로젝트 구조 파악 및 파일 생성/수정이 추가로 필요할 수 있습니다.

## 관련 가능 파일 (Files Likely Involved)
- apps/posts/models.py
- apps/posts/forms.py
- apps/posts/tests.py
- templates/posts/post_list.html
- templates/posts/post_detail.html
- templates/posts/post_form.html
- (추가적으로 views.py, urls.py 등)

## 제안된 접근 방식 (Suggested Approach)
1. Post 모델에 visibility 필드(CharField with choices) 추가 및 마이그레이션
2. PostForm에 visibility 필드 추가
3. 템플릿에 공개 범주 선택 UI 및 표시 뱃지 추가
4. PostListView, PostDetailView에서 접근 권한 로직 구현
5. 테스트 코드 작성 및 통과 확인

## 멘토링 노트 (Men

### 코드 해설 🧩

- TOOLS 리스트는 세션에 노출할 도구의 전체 목록입니다.
- SYSTEM_PROMPT는 에이전트의 역할, 절차, 응답 형식을 강하게 유도합니다.
- 이벤트 핸들러는 메시지뿐 아니라 어떤 도구가 실행되는지도 보여줍니다.
- 이 구조 덕분에 사용자는 에이전트가 단순히 답을 생성한 것이 아니라, 실제로 근거를 수집했다는 점을 확인할 수 있습니다.

터미널 실행 예시:

**python app_final.py taeyo-kim MyDemo 78**

## 6단계. FastAPI와 SSE로 웹 UI 연결하기 🌐

이번에는 작성된 에이전트를 브라우저에 연결합니다.   
프론트엔드는 현 저장소의 /src 디렉토리에 이미 준비되어 있기에, 백엔드에서 SSE 이벤트를 흘려주기만 하면 됩니다.

### 알아둘 점

- FastAPI는 웹 서버 역할을 합니다.
- StreamingResponse는 SSE 스트림을 브라우저로 보냅니다.
- 내부적으로는 asyncio.Queue를 사용해 이벤트를 순서대로 전달합니다.

In [ ]:
# 이번 코드의 실제 실행은 터미널에서 수행합니다.
# 노트북에서 실행할 경우 에러가 발생할 수 있습니다.
# 이 코드는 이해를 돕기 위해 노트북에 복사해 둔 버전입니다.

# FastAPI 애플리케이션 객체를 만들기 위해 클래스를 가져옵니다.
from fastapi import FastAPI
# 정적 HTML 파일 응답과 스트리밍 응답에 사용할 클래스를 가져옵니다.
from fastapi.responses import FileResponse, StreamingResponse
# 정적 파일 디렉터리를 마운트하기 위한 클래스를 가져옵니다.
from fastapi.staticfiles import StaticFiles

# 웹 서버의 기본 앱 객체를 생성하고 제목을 지정합니다.
app = FastAPI(title="GitHub Issue Complexity Analyser")

# 현재 작업 디렉터리 기준으로 정적 파일 폴더 경로를 계산합니다.
static_dir = Path.cwd() / "src" / "static"
# /static 경로로 CSS, JS, 이미지 같은 정적 자원을 제공하도록 연결합니다.
app.mount("/static", StaticFiles(directory=static_dir), name="static")


# 루트 경로 요청이 오면 메인 HTML 페이지를 반환합니다.
@app.get("/")
async def root():
    # 정적 디렉터리 안의 index.html 파일을 그대로 내려줍니다.
    return FileResponse(static_dir / "index.html")

# 서버가 살아 있는지 확인하는 간단한 헬스 체크 엔드포인트입니다.
@app.get("/health")
async def health():
    # 프런트엔드나 배포 환경이 서버 상태를 쉽게 확인할 수 있게 JSON을 반환합니다.
    return {"status": "healthy"}

# SDK 이벤트에 들어 있는 인자 값을 dict 형태로 정규화하는 헬퍼 함수입니다.
def _parse_args(raw):
    # 이미 dict이면 그대로 반환합니다.
    if isinstance(raw, dict):
        return raw
    # 문자열이면 JSON으로 파싱을 시도합니다.
    if isinstance(raw, str):
        try:
            # JSON 문자열을 파이썬 dict로 변환합니다.
            return json.loads(raw)
        except (json.JSONDecodeError, TypeError):
            # 파싱에 실패하면 빈 dict를 반환합니다.
            return {}
    # pydantic 객체처럼 model_dump를 지원하면 dict로 변환합니다.
    if hasattr(raw, "model_dump"):
        return raw.model_dump()
    # 어떤 형식도 아니면 안전하게 빈 dict를 반환합니다.
    return {}


# GitHub 이슈 분석 과정을 SSE 이벤트로 흘려보내는 비동기 제너레이터입니다.
async def stream_analysis(owner: str, repo: str, issue_number: int):
    """브라우저로 SSE 이벤트를 스트리밍하는 제너레이터"""
    # Copilot 백엔드와 통신할 클라이언트를 생성합니다.
    client = CopilotClient()
    # 세션을 만들기 전에 클라이언트를 시작합니다.
    await client.start()

    # GitHub API 호출과 세션 인증에 사용할 토큰을 읽습니다.
    token = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
    # 모델, 도구, 권한 정책, GitHub 토큰을 포함한 세션을 생성합니다.
    session = await client.create_session(
        # 이번 예제에서 사용할 모델 이름입니다.
        model="gpt-4.1",
        # 에이전트가 사용할 도구 목록을 세션에 연결합니다.
        tools=TOOLS,
        # 데모에서는 도구 실행 권한을 자동 승인합니다.
        on_permission_request=PermissionHandler.approve_all,
        # GitHub 토큰을 세션에 전달합니다.
        github_token=token,
    )

    # SDK 이벤트를 SSE 응답 루프에 전달하기 위한 비동기 큐를 만듭니다.
    queue = asyncio.Queue()

    # 세션에서 발생하는 이벤트를 큐에 적재할 콜백 함수를 정의합니다.
    def on_event(event):
        # 이벤트 타입을 문자열로 정규화해 비교하기 쉽게 만듭니다.
        event_name = event.type.value if hasattr(event.type, "value") else str(event.type)

        # 모델이 일반 텍스트 메시지를 생성하면 message 이벤트로 큐에 넣습니다.
        if event_name == "assistant.message":
            # 이벤트 데이터에서 텍스트 내용을 안전하게 꺼냅니다.
            content = getattr(event.data, "content", "")
            # 공백만 있는 메시지는 제외하고 실제 텍스트만 전달합니다.
            if content and content.strip():
                queue.put_nowait(("message", content))
        # 도구 실행이 시작되면 도구 이름과 인자를 큐에 넣습니다.
        elif event_name == "tool.execution_start":
            # 실행 중인 도구 이름을 이벤트 데이터에서 읽습니다.
            tool_name = getattr(event.data, "tool_name", None)
            # 도구 인자는 dict 형태로 정규화합니다.
            args = _parse_args(getattr(event.data, "arguments", None))
            # 도구 이름이 있으면 프런트엔드에 표시할 이벤트를 큐에 넣습니다.
            if tool_name:
                queue.put_nowait(("tool_call", {"name": tool_name, "args": args}))
        # 세션이 idle 상태가 되면 이번 응답이 끝났다는 뜻입니다.
        elif event_name == "session.idle":
            # 종료 신호를 큐에 넣어 SSE 루프를 마무리합니다.
            queue.put_nowait(("done", None))

    # 메시지를 보내기 전에 이벤트 핸들러를 세션에 등록합니다.
    session.on(on_event)
    # 시스템 프롬프트와 실제 분석 대상 이슈 정보를 함께 전달합니다.
    await session.send(
        f"{SYSTEM_PROMPT}\n\nPlease analyse GitHub issue #{issue_number} in {owner}/{repo}."
    )

    # 큐에서 이벤트를 하나씩 꺼내 SSE 포맷 문자열로 변환해 브라우저에 전달합니다.
    while True:
        # 다음 SDK 이벤트가 들어올 때까지 기다립니다.
        event_type, data = await queue.get()
        # 일반 메시지는 message 이벤트로 전송합니다.
        if event_type == "message":
            yield f"event: message\ndata: {json.dumps({'content': data})}\n\n"
        # 도구 호출은 tool_call 이벤트로 전송합니다.
        elif event_type == "tool_call":
            yield f"event: tool_call\ndata: {json.dumps(data)}\n\n"
        # 완료 신호가 오면 done 이벤트를 보낸 뒤 루프를 끝냅니다.
        elif event_type == "done":
            yield f"event: done\ndata: {json.dumps({'status': 'complete'})}\n\n"
            break

    # 스트리밍이 끝나면 세션 연결을 정리합니다.
    await session.disconnect()
    # 클라이언트도 종료해 리소스를 해제합니다.
    await client.stop()


# 브라우저가 호출할 SSE 엔드포인트를 정의합니다.
@app.get("/analyse/stream")
async def analyse_stream(owner: str, repo: str, issue_number: int):
    # stream_analysis 제너레이터를 text/event-stream 형식으로 감싸 반환합니다.
    return StreamingResponse(
        stream_analysis(owner, repo, issue_number),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "Connection": "keep-alive"},
    )

### 코드 해설 📡

- queue는 SDK 이벤트와 HTTP 스트림 사이의 완충지대 역할을 합니다.
- assistant.message는 일반 메시지 이벤트, tool.execution_start는 도구 시작 이벤트입니다.
- StreamingResponse는 제너레이터가 yield하는 문자열을 그대로 브라우저에 전달합니다.
- 이 방식 덕분에 프론트엔드는 에이전트의 답변과 도구 호출을 실시간 채팅처럼 렌더링할 수 있습니다.

### 실행 방법

실제 실행은 터미널에서 다음의 명령을 수행합니다.

**python app_final.py serve**

실행 후, http://localhost:8000/에 접속하면 웹 UI에서 에이전트의 분석 과정을 실시간으로 볼 수 있습니다.
테스트 URL로는 다음의 경로를 입력해 봅니다.  

https://github.com/taeyo-kim/MyDemo/issues/78

<img src="./images/sdk01.png" width="800" />


## 7단계. GitHub에 분석 결과 다시 쓰기 ✍️

이제 분석 결과를 사람이 검토한 뒤, 버튼 클릭으로 GitHub 이슈에 코멘트와 난이도 라벨을 남깁니다.
이 단계의 핵심은 Human-in-the-loop입니다.
즉, 에이전트가 자동으로 바로 쓰지 않고, 사용자가 결과를 보고 승인한 뒤에만 기록합니다. ✅

다만, 이 코드가 올바로 동작하기 위해서는 본인의 Repository에 존재하는 Issue를 대상으로 수행해야 합니다. 본인의 저장소가 아니면 Write 권한이 없기에 올바로 동작하지 않습니다.

그렇기에, 실습에서는 자신의 GitHub 리포지토리에 새로운 Issue를 생성하거나, 혹은 기존에 존재하는 Issue를 대상으로 앞의 과정을 수행해야 합니다. Issue를 만드는 것이 여의치 않다면 실습없이 코드만 이해하고 넘어가도 됩니다.

관련 코드는 다음과 같습니다. 또한, 이 코드는 이미 최종 파일인 app_final.py에 반영되어 있습니다. 앞의 과정에서 에이전트가 분석을 마치고 결과를 반환하면, 제일 하단에 [Post to GitHub Issue] 버튼을 눌러서 분석 결과를 이슈에 추가합니다.

<img src="./images/sdk02.png" width="800" />

In [ ]:
SKILL_LABELS = {
    "junior": ["good first issue", "difficulty: easy"],
    "mid-level": ["difficulty: medium"],
    "senior": ["difficulty: hard"],
    "senior+": ["difficulty: expert"],
}


async def post_comment(owner: str, repo: str, issue_number: int, body: str):
    import httpx

    token = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
    async with httpx.AsyncClient() as http:
        response = await http.post(
            f"https://api.github.com/repos/{owner}/{repo}/issues/{issue_number}/comments",
            headers={
                "Authorization": f"Bearer {token}",
                "Accept": "application/vnd.github+json",
            },
            json={"body": body},
        )
        response.raise_for_status()


async def add_labels(owner: str, repo: str, issue_number: int, labels: list[str]):
    import httpx

    token = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
    async with httpx.AsyncClient() as http:
        response = await http.post(
            f"https://api.github.com/repos/{owner}/{repo}/issues/{issue_number}/labels",
            headers={
                "Authorization": f"Bearer {token}",
                "Accept": "application/vnd.github+json",
            },
            json={"labels": labels},
        )
        response.raise_for_status()


class PostAnalysisRequest(BaseModel):
    owner: str
    repo: str
    issue_number: int
    body: str


@app.post("/post-analysis")
async def post_analysis(req: PostAnalysisRequest):
    import re

    await post_comment(req.owner, req.repo, req.issue_number, req.body)

    match = re.search(r"recommended skill level.*", req.body, re.IGNORECASE)
    level_line = match.group(0).lower() if match else ""

    for level in sorted(SKILL_LABELS, key=len, reverse=True):
        if level in level_line:
            await add_labels(req.owner, req.repo, req.issue_number, SKILL_LABELS[level])
            break

    return {"status": "posted"}


def parse_github_url(url: str) -> tuple[str, str, int]:
    parts = url.rstrip("/").replace("https://github.com/", "").split("/")
    if len(parts) >= 4 and parts[2] == "issues":
        return parts[0], parts[1], int(parts[3])
    raise ValueError(f"Invalid GitHub issue URL: {url}")


async def validate_tool_args(event):
    """위험한 파일 경로를 차단하는 예시 훅"""
    if event.data.tool_name == "get_file_content":
        path = event.data.arguments.get("path", "")
        if ".." in path or path.startswith("/") or path.startswith("~"):
            return {"decision": "reject", "message": "Blocked: unsafe path"}
        sensitive = [".env", ".git/", "secrets", "credentials", "token"]
        if any(item in path.lower() for item in sensitive):
            return {"decision": "reject", "message": "Blocked: sensitive file"}
    return {"decision": "allow"}

### 코드 해설 🔐

- post_comment는 분석 결과를 GitHub 이슈 코멘트로 남깁니다.
- add_labels는 추천 난이도에 맞는 라벨을 이슈에 추가합니다.
- post_analysis 엔드포인트는 이미 화면에 표시된 분석 결과를 다시 받아서 기록합니다. 즉, 서버가 분석을 재실행하지 않습니다.
- validate_tool_args는 잠재적으로 위험한 도구 호출을 사전에 차단하는 훅 예시입니다.

실전에서는 여기에 더해 재시도 정책, 로깅, 타임아웃, 최대 도구 호출 횟수 제한도 함께 고려해야 합니다. 🛡️

## 8단계. 마무리 정리와 실습 과제 ✅

### 오늘 만든 것

- Copilot SDK 세션 생성
- 스트리밍 이벤트 처리
- GitHub API 도구 4종 구성
- 시스템 프롬프트 기반 이슈 분석기
- FastAPI + SSE 웹 스트리밍
- GitHub 코멘트/라벨 쓰기
- 간단한 안전 훅

### 직접 확장해볼 아이디어 💡

1. 분석 결과를 Markdown 대신 JSON 스키마로 고정해보기
2. 파일 읽기 도구에 더 강한 경로 검증 넣기
3. 특정 라벨이나 이슈 템플릿을 자동으로 고려하도록 프롬프트 개선하기
4. 분석 결과를 데이터베이스에 저장해서 이력 비교 기능 만들기
5. 하나의 세션에서 후속 질문까지 받는 멀티턴 UI로 확장하기
